# 8. ChIP-seq occupancy of core-motif sites (Fig. 4d)

Are the core motifs bound? For every TF ChIP-seq experiment of five ENCODE cell lines (K562, HepG2, GM12878, MCF-7, A549; 1,151 TF-line pairs), the fraction of the TF's TSS-proximal peaks (summit ± 150 bp within TSS ± 10 kb) that contain a site of a core motif is compared with the same fraction for the ten random control sets of 43 JASPAR motifs.

**ENCODE peaks are not redistributed.** `data/encode/encode_tf_chipseq_peaks.tsv` lists the 1,348 IDR-thresholded peak files (six cell lines; the sixth, HEK293, is for notebooks 10 and 11) and `bin/download_encode_peaks.py` downloads them (1 GB, 20-40 min) into `$ENHANCER_ROOT/encode4_tf_chipseq/{cell}/{TF}_human/{ENCFF}.bed`:
```
python bin/download_encode_peaks.py --dest $ENHANCER_ROOT/encode4_tf_chipseq
```
The FIMO scans of the peaks (43 core motifs + the 291-motif control pool, 2-7 CPU-hours per cell line) are released instead of rerun: Zenodo `fig4_fig5_motif_sites_on_peaks.tar.gz` unpacked under `$ENHANCER_ROOT`.

In [ ]:
# ---- Shared setup (identical in every notebook) --------------------------------------------------------
import os, sys, json, subprocess, time, glob, shutil
REPO = os.path.abspath(os.path.join(os.getcwd(), ".."))          # notebooks/ lives one level below the repository root
CODE = os.path.join(REPO, "code")
roots = json.load(open(os.path.join(REPO, ".roots.json")))          # written by bin/rewrite_paths.py (see PATHS.md)
ENHANCER_ROOT, SEQ_ROOT = roots["enhancer_root"], roots["seq_root"]
def run(cmd, cwd=None, env=None):
    """Run one repository script from the right folder and stream its output here. Stops the notebook if it fails."""
    print("$", cmd, "   (in", cwd or os.getcwd(), ")"); t = time.time()
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, executable="/bin/bash", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    for line in p.stdout: print(line, end="")
    p.wait(); print(f"[exit {p.returncode}, {time.time()-t:.0f} s]")
    assert p.returncode == 0, "this step failed - read the output above"
def n_motifs(meme):
    """Number of MOTIF entries in a MEME file."""
    return sum(1 for l in open(meme) if l.startswith("MOTIF")) if os.path.exists(meme) else None
def show_logos(meme, n=5, title=""):
    """Sequence logos of the first n motifs of a MEME file (logomaker)."""
    import numpy as np, pandas as pd, matplotlib.pyplot as plt, logomaker
    blocks = open(meme).read().split("MOTIF ")[1:n+1]; fig, axes = plt.subplots(1, len(blocks), figsize=(2.2*len(blocks), 1.4))
    for ax, b in zip(np.atleast_1d(axes), blocks):
        rows = [l.split() for l in b.split("\n")[2:] if l.strip() and not l.startswith("letter") and not l.startswith("URL")]
        pwm = pd.DataFrame([[float(x) for x in r] for r in rows if len(r) == 4], columns=list("ACGT"))
        logomaker.Logo(logomaker.transform_matrix(pwm, from_type="probability", to_type="information"), ax=ax); ax.set_title(b.split("\n")[0], fontsize=8); ax.set_yticks([]); ax.set_xticks([])
    fig.suptitle(title, fontsize=9); plt.tight_layout(); plt.show()
print("repository:", REPO); print("ENHANCER_ROOT:", ENHANCER_ROOT); print("SEQ_ROOT:", SEQ_ROOT)


In [ ]:
ENCODE = f"{ENHANCER_ROOT}/encode4_tf_chipseq"     # bin/download_encode_peaks.py --dest $ENHANCER_ROOT/encode4_tf_chipseq
def encode_status(cells):
    for c in cells:
        n = len(glob.glob(f"{ENCODE}/{c}/*/*.bed")); want = sum(1 for l in open(f"{REPO}/data/encode/encode_tf_chipseq_peaks.tsv") if l.split("\t")[0] == c)
        print(f"  {'ok ' if n == want else '-- '} {c}: {n} of {want} ENCODE peak files")
CE = f"{CODE}/chipseq_enrichment"; F4 = f"{REPO}/figures/fig4"; WORK = f"{ENHANCER_ROOT}/notebook_runs/fig4"; os.makedirs(WORK, exist_ok=True)
CELLS = ["HepG2", "K562", "GM12878", "MCF.7", "A549"]
import numpy as np, pandas as pd
from IPython.display import Image, display
encode_status(CELLS + ["HEK293"])
for c in CELLS: print(f"  {'ok ' if os.path.exists(f'{CE}/results/fimo_poolscan/{c}/fimo_peaks/fimo.tsv') else '-- '} motif sites on {c} peaks (Zenodo)")
for f in glob.glob(f"{CE}/results/oldpool10/fig4d_*.csv"): shutil.copy(f, f"{WORK}/{os.path.basename(f).replace('.csv', '.published.csv')}")

## 8.1 TSS-proximal peaks of every TF
`01_prepare_peaks.py` reads every TF's narrowPeak file, takes summit ± 150 bp, keeps the windows overlapping TSS ± 10 kb (`data/genes/hg38_promoters_cage_corrected.bed`), and writes per cell line `data/{cell}/unique_summits.bed`, `tf_peak_sets.pkl` (TF -> its peak ids) and `tf_names.txt`. A few minutes. HEK293 is prepared here too for notebook 11.

In [ ]:
run("python 01_prepare_peaks.py " + " ".join(CELLS + ["HEK293"]), cwd=CE)
for c in CELLS + ["HEK293"]:
    n = sum(1 for _ in open(f"{CE}/data/{c}/tf_names.txt")); w = sum(1 for _ in open(f"{CE}/data/{c}/unique_summits.bed")); print(f"  {c}: {n} TFs, {w:,} TSS-proximal peak windows")

## 8.2 Fraction of peaks with a site, core versus control
The released scans give every motif site in those peak windows (`fimo_poolscan`: core + control pool; `fimo_poolscan_extra`: 13 further control motifs). `09_pool_enrichment.py --stage1-only` computes, per cell line, TF and motif, the fraction of the TF's peaks with a site; `09b_extra_stage1.py` the same for the extra motifs; `10_oldpool_10sets.py` pools the five lines and scores the ten control sets against the 43 core motifs -> `results/oldpool10/fig4d_{permotif,per_cell,stats}.csv`.

In [ ]:
run("python 09_pool_enrichment.py --stage1-only", cwd=CE); run("python 09b_extra_stage1.py", cwd=CE); run("python 10_oldpool_10sets.py", cwd=CE)
st = pd.read_csv(f"{CE}/results/oldpool10/fig4d_stats.csv"); print(st.round(4).to_string(index=False))
pc = pd.read_csv(f"{CE}/results/oldpool10/fig4d_per_cell.csv"); old = pd.read_csv(f"{WORK}/fig4d_per_cell.published.csv")
print("\nper cell line (core median % / control mean %):"); print(pc[["cell", "core_median", "ctrl_mean", "ctrl_sd"]].round(3).to_string(index=False))
same = pc.round(6).equals(old.round(6))
print("\nidentical to the published table:", same)
if not same:
    cmp = pc.merge(old, on="cell", suffixes=("", "_pub"))
    d = (cmp[["core_median", "ctrl_mean"]].values - cmp[["core_median_pub", "ctrl_mean_pub"]].values)
    worst = cmp.cell[abs(d).max(axis=1).argmax()]
    print(f"  largest difference {abs(d).max():.3f} percentage points (in {worst}); fold change differs by at most "
          f"{abs(pc.set_index('cell').core_median/pc.set_index('cell').ctrl_mean - old.set_index('cell').core_median/old.set_index('cell').ctrl_mean).max():.4f}")
    print("  cause: ENCODE has since released one further TF experiment for K562 and for HepG2 than the published run used")
    print("  (data/encode/encode_tf_chipseq_peaks.tsv lists what is downloaded today, 352 and 550; the published run used 351 and 549)")

## 8.3 Panel d

In [ ]:
run("python -c \"import make_fig4_panels as P; P.panel_4d()\"", cwd=F4); display(Image(f"{F4}/fig4d_chipseq_occupancy.png", width=520))